# 00 · Explore the history

A first look at the historical data: what are we working with? Reads the local Parquet (`data/raw/rawtree_history`, from `python -m training.data.export_history`) through DuckDB. Views: `catalog`, `sessions` (carousels), `events` (organic activity). For single rows, use the CLI: `python -m training.data.inspect product|user|carousel <id>`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root, when opened from analysis/

import matplotlib.pyplot as plt
from training.data.paths import con, history_dir

c = con()
print(history_dir())
c.sql("SELECT 'catalog' t, count(*) n FROM catalog UNION ALL SELECT 'sessions', count(*) FROM sessions UNION ALL SELECT 'events', count(*) FROM events")

## Volume per day

In [ ]:
daily = c.sql("""
  SELECT d, sum(carousels) carousels, sum(views) views, sum(carts) carts, sum(purchases) purchases FROM (
    SELECT ts::DATE d, count(*) carousels, 0 views, 0 carts, 0 purchases FROM sessions GROUP BY d
    UNION ALL
    SELECT ts::DATE, 0, count_if(event_type='view'), count_if(event_type='add_to_cart'), count_if(event_type='purchase') FROM events GROUP BY 1
  ) GROUP BY d ORDER BY d""").df()
daily.set_index('d').plot(subplots=True, figsize=(12, 8), title='per day'); plt.tight_layout()

## Context mix (device · region · page) and carousel outcomes

In [ ]:
c.sql("""SELECT device, page, count(*) carousels, round(avg((clicked IS NOT NULL)::INT), 4) ctr,
       round(avg((purchased IS NOT NULL)::INT), 4) purchase_rate
FROM sessions GROUP BY ALL ORDER BY carousels DESC""").df()

In [ ]:
c.sql("""SELECT region, count(*) carousels, round(avg((clicked IS NOT NULL)::INT), 4) ctr,
       round(avg((purchased IS NOT NULL)::INT), 4) purchase_rate FROM sessions GROUP BY ALL ORDER BY carousels DESC""").df()

## Purchases per category (organic vs. carousel)

In [ ]:
c.sql("""
WITH organic AS (SELECT c.category, count(*) n FROM events e JOIN catalog c USING (product_id)
                 WHERE event_type = 'purchase' GROUP BY ALL),
     carousel AS (SELECT c.category, count(*) n FROM sessions s JOIN catalog c ON c.product_id = s.purchased GROUP BY ALL),
     cat AS (SELECT category, count(*) products, round(median(price_eur), 2) median_price FROM catalog GROUP BY ALL)
SELECT category, products, median_price, organic.n organic_purchases, carousel.n carousel_purchases
FROM cat LEFT JOIN organic USING (category) LEFT JOIN carousel USING (category) ORDER BY organic_purchases DESC""").df()

## Prices

In [ ]:
prices = c.sql("SELECT price_eur FROM catalog").df()
bought = c.sql("SELECT price_eur FROM events WHERE event_type = 'purchase'").df()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
prices.price_eur.plot.hist(bins=60, ax=ax[0], logy=True, title='catalogue prices')
bought.price_eur.plot.hist(bins=60, ax=ax[1], logy=True, title='prices of organic purchases'); plt.tight_layout()

## Position effect in the carousel

In [ ]:
c.sql("""
WITH s AS (SELECT unnest(range(1, 11)) slot, unnest(shown) pid, clicked, purchased FROM sessions)
SELECT slot, round(avg((clicked = pid)::INT), 5) ctr, round(avg((purchased = pid)::INT), 5) purchase_rate FROM s GROUP BY slot ORDER BY slot""").df()